# LogSentinel single-GPU training
Open this notebook from the root of a LogSentinel checkout mounted in Colab. The notebook validates the runtime, prepares a bounded public-data experiment, and starts QLoRA training only when the final training cell is run.

In [ ]:
%pip install -q -e '.[data,ml,dev]'

In [ ]:
from pathlib import Path

import torch

assert Path('pyproject.toml').is_file(), (
    'Change the working directory to the LogSentinel repository root.'
)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## Prepare and validate
Use a bounded pass first. Increase the limit only after storage, class balance, and sequence counts have been inspected.

In [ ]:
from subprocess import run

commands = [
    ['logsentinel', 'prepare', '--dataset', 'hdfs', '--limit', '200000',
     '--output', 'data/processed/hdfs-colab.json'],
    ['logsentinel', 'train-baselines', '--prepared',
     'data/processed/hdfs-colab.json', '--output',
     'reports/generated/hdfs-colab-baselines.json'],
    ['logsentinel', 'train-transformer', '--prepared',
     'data/processed/hdfs-colab.json', '--output',
     'reports/generated/hdfs-colab-transformer-plan.json', '--dry-run'],
]
for command in commands:
    run(command, check=True)

## Train the adapter
The following cell downloads Qwen2.5-1.5B and performs 4-bit QLoRA training. Run it only on a CUDA runtime. It writes adapter weights but no evaluation metrics.

In [ ]:
from subprocess import run

assert torch.cuda.is_available(), 'Select a Colab GPU runtime before training.'
run(
    ['logsentinel', 'train-transformer', '--prepared',
     'data/processed/hdfs-colab.json', '--output',
     'reports/generated/hdfs-colab-training.json', '--adapter-output',
     'artifacts/adapters/hdfs-qwen-v1', '--no-dry-run'],
    check=True,
)